# F1 Qualifying Time Prediction — Mini Project
**หัวข้อ:** ทำนายเวลา Qualifying ของนักแข่ง F1 (Regression)

**สถานะปัจจุบัน:** อยู่ระหว่างขั้นตอนที่ 2 (Dataset) — ดึงข้อมูลทุกสนามของปี 2021-2023 จาก FastF1 API

**ลำดับการรัน:** รันเรียงจากบนลงล่างตามลำดับ ห้ามข้าม

## Cell 1: Setup

In [1]:
import fastf1
import pandas as pd
import numpy as np
import os
import time

os.makedirs('f1_cache', exist_ok=True)
fastf1.Cache.enable_cache('f1_cache')

print("Setup สำเร็จ")


Setup สำเร็จ


## Cell 2: ฟังก์ชันดึงข้อมูล (เวอร์ชันล่าสุด — เร็ว + มี Circuit + เก็บ skip_log)

**หมายเหตุสำคัญ:** ปิด `telemetry=False` เพื่อไม่ให้โหลดข้อมูลตำแหน่ง/ความเร็วรถแบบละเอียด (ซึ่งไม่ได้ใช้ในโปรเจกต์นี้ แต่ทำให้ช้ามาก — จาก 36 นาที/สนาม เหลือ ~1-2 วินาที/สนาม)

In [2]:
skip_log = []  # เก็บสนามที่ดึงไม่สำเร็จ พร้อมเหตุผล

def get_practice_best_laps(year, gp, session_code):
    try:
        session = fastf1.get_session(year, gp, session_code)
        session.load(telemetry=False, weather=False, laps=True, messages=False)
        laps = session.laps
        best = laps.groupby('Driver')['LapTime'].min().reset_index()
        best.columns = ['Driver', f'{session_code}_Time']
        best[f'{session_code}_Time'] = best[f'{session_code}_Time'].dt.total_seconds()
        return best
    except Exception as e:
        skip_log.append((gp, year, session_code, str(e)))
        return pd.DataFrame(columns=['Driver', f'{session_code}_Time'])


def get_quali_data(year, gp):
    print(f"กำลังดึงข้อมูล: {gp} {year}")

    quali = fastf1.get_session(year, gp, 'Q')
    quali.load(telemetry=False, weather=True, laps=False, messages=False)

    results = quali.results[['Abbreviation', 'TeamName', 'Q1', 'Q2', 'Q3']].copy()
    results = results.rename(columns={'Abbreviation': 'Driver', 'TeamName': 'Team'})

    for col in ['Q1', 'Q2', 'Q3']:
        results[col] = results[col].dt.total_seconds()

    results['QualiTime'] = results[['Q1', 'Q2', 'Q3']].min(axis=1)
    # เก็บ Q1/Q2/Q3 และแถวที่ QualiTime ว่างไว้ใน raw CSV เพื่อให้ขั้น cleaning จัดการ

    weather = quali.weather_data
    results['AirTemp'] = weather['AirTemp'].mean()
    results['TrackTemp'] = weather['TrackTemp'].mean()
    results['Humidity'] = weather['Humidity'].mean()
    results['Rainfall'] = weather['Rainfall'].mean()

    for fp in ['FP1', 'FP2', 'FP3']:
        fp_data = get_practice_best_laps(year, gp, fp)
        results = results.merge(fp_data, on='Driver', how='left')

    results['Year'] = year
    results['Circuit'] = gp
    return results


def get_season_data(year):
    """ดึงข้อมูล Qualifying ทุกสนามของปีที่กำหนด"""
    schedule = fastf1.get_event_schedule(year)
    schedule = schedule[schedule['EventFormat'] != 'testing']

    season_data = []
    for gp in schedule['EventName']:
        try:
            df_gp = get_quali_data(year, gp)
            season_data.append(df_gp)
        except Exception as e:
            skip_log.append((gp, year, 'Qualifying', str(e)))
            print(f"  [Skip] {gp} {year}: {e}")
            continue

    return pd.concat(season_data, ignore_index=True) if season_data else pd.DataFrame()


def patch_missing_circuits(year, df_existing):
    """เช็คว่าสนามไหนของปีนี้ยังขาดอยู่ แล้วดึงมาเติมให้ครบ
    ใช้กรณีที่ดึงข้อมูลค้างกลางทาง (เช่น ปิดคอม/ถูกขัดจังหวะ)"""
    schedule = fastf1.get_event_schedule(year)
    schedule = schedule[schedule['EventFormat'] != 'testing']
    all_circuits = set(schedule['EventName'])
    got_circuits = set(df_existing['Circuit'].unique()) if 'Circuit' in df_existing.columns else set()
    missing = all_circuits - got_circuits

    if not missing:
        print(f"ปี {year}: ครบทุกสนามแล้ว ({len(got_circuits)}/{len(all_circuits)})")
        return df_existing

    print(f"ปี {year}: ขาด {len(missing)} สนาม -> {sorted(missing)}")
    additional = []
    for gp in missing:
        try:
            df_gp = get_quali_data(year, gp)
            additional.append(df_gp)
        except Exception as e:
            skip_log.append((gp, year, 'Qualifying', str(e)))
            print(f"  [Skip] {gp} {year}: {e}")

    if additional:
        df_combined = pd.concat([df_existing] + additional, ignore_index=True)
    else:
        df_combined = df_existing

    print(f"ปี {year}: ตอนนี้ได้ {df_combined['Circuit'].nunique()}/{len(all_circuits)} สนาม")
    return df_combined


## Cell 3: ดึงข้อมูลปี 2021 (ทุกสนาม)

In [ ]:
df_2021 = get_season_data(2021)
print(f"\nปี 2021 ได้ข้อมูล: {df_2021.shape[0]} แถว, {df_2021['Circuit'].nunique()} สนาม")
df_2021.to_csv('f1_2021_raw.csv', index=False)


core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


กำลังดึงข้อมูล: Bahrain Grand Prix 2021


req            INFO 	Using cached data for weather_data
core           INFO 	Finished loading data for 20 drivers: ['33', '44', '77', '16', '10', '3', '4', '55', '14', '18', '11', '99', '22', '7', '63', '31', '6', '5', '47', '9']
core           INFO 	Loading data for Bahrain Grand Prix - Practice 1 [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core           INFO 	Finished loading data for 20 drivers: ['3', '4', '5', '6', '7', '9', '10', '11', '14', '16', '18', '22', '31', '33', '44', '47', '55', '63', '77', '99']
core           INFO 	Loading data for Bahrain Grand Prix - Practice 2 [v3.8.3]
req            INFO 	Using cache

กำลังดึงข้อมูล: Emilia Romagna Grand Prix 2021


req            INFO 	Using cached data for weather_data
core           INFO 	Finished loading data for 20 drivers: ['44', '11', '33', '16', '10', '3', '4', '77', '31', '18', '55', '63', '5', '6', '14', '7', '99', '47', '9', '22']
core           INFO 	Loading data for Emilia Romagna Grand Prix - Practice 1 [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	No tyre data for driver 3
core        WARNING 	No tyre data for driver 4
core        WARNING 	No tyre data for driver 5
core        WARNING 	No tyre data for driver 6
core        WARNING 	No tyre data for driver 7
core        WARNING 	No tyre data for drive

...

req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for weather_data. Loading data...
_api           INFO 	Fetching weather data...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 20 drivers: ['44', '77', '33', '16', '10', '14', '11', '4', '18', '22', '5', '31', '63', '47', '55', '3', '6', '99', '7', '9']
core           INFO 	Loading data for Turkish Grand Prix - Practice 1 [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to 

กำลังดึงข้อมูล: United States Grand Prix 2021


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for weather_data. Loading data...
_api           INFO 	Fetching weather data...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 20 drivers: ['33', '44', '11', '77', '16', '55', '3', '4', '10', '22', '31', '5', '99', '14', '63', '18', '6', '7', '47', '9']
core           INFO 	Loading data for United States Grand Prix - Practice 1 [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been writt

กำลังดึงข้อมูล: Mexico City Grand Prix 2021


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for weather_data. Loading data...
_api           INFO 	Fetching weather data...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 20 drivers: ['77', '44', '33', '11', '10', '55', '3', '16', '22', '4', '5', '7', '63', '99', '31', '14', '6', '47', '9', '18']
core           INFO 	Loading data for Mexico City Grand Prix - Practice 1 [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written

### Cell 3b: เช็ค + เติมสนามที่ขาด (เผื่อโดนขัดจังหวะกลางทาง) แล้ว save ทับ

In [ ]:
df_2021 = pd.read_csv('f1_2021_raw.csv')
df_2021 = patch_missing_circuits(2021, df_2021)
df_2021.to_csv('f1_2021_raw.csv', index=False)
print(f"บันทึกแล้ว: {df_2021.shape[0]} แถว, {df_2021['Circuit'].nunique()} สนาม")


## Cell 4: ดึงข้อมูลปี 2022 (ทุกสนาม)

In [ ]:
df_2022 = get_season_data(2022)
print(f"\nปี 2022 ได้ข้อมูล: {df_2022.shape[0]} แถว, {df_2022['Circuit'].nunique()} สนาม")
df_2022.to_csv('f1_2022_raw.csv', index=False)


### Cell 4b: เช็ค + เติมสนามที่ขาด แล้ว save ทับ

In [ ]:
df_2022 = pd.read_csv('f1_2022_raw.csv')
df_2022 = patch_missing_circuits(2022, df_2022)
df_2022.to_csv('f1_2022_raw.csv', index=False)
print(f"บันทึกแล้ว: {df_2022.shape[0]} แถว, {df_2022['Circuit'].nunique()} สนาม")


## Cell 5: ดึงข้อมูลปี 2023 (ทุกสนาม)

In [ ]:
df_2023 = get_season_data(2023)
print(f"\nปี 2023 ได้ข้อมูล: {df_2023.shape[0]} แถว, {df_2023['Circuit'].nunique()} สนาม")
df_2023.to_csv('f1_2023_raw.csv', index=False)


### Cell 5b: เช็ค + เติมสนามที่ขาด แล้ว save ทับ

In [ ]:
df_2023 = pd.read_csv('f1_2023_raw.csv')
df_2023 = patch_missing_circuits(2023, df_2023)
df_2023.to_csv('f1_2023_raw.csv', index=False)
print(f"บันทึกแล้ว: {df_2023.shape[0]} แถว, {df_2023['Circuit'].nunique()} สนาม")


## Cell 6: รวมข้อมูลทั้ง 3 ปีเป็นตารางเดียว

In [ ]:
df_2021 = pd.read_csv('f1_2021_raw.csv')
df_2022 = pd.read_csv('f1_2022_raw.csv')
df_2023 = pd.read_csv('f1_2023_raw.csv')

df_final = pd.concat([df_2021, df_2022, df_2023], ignore_index=True)
df_final.to_csv('f1_all_circuits_raw.csv', index=False)

print(f"รวมข้อมูลทั้งหมด: {df_final.shape[0]} แถว, {df_final.shape[1]} คอลัมน์")
print(f"จำนวนสนามรวม (นับซ้ำได้ต่อปี): {df_final.groupby('Year')['Circuit'].nunique()}")
df_final.head()


## Cell 7: เช็ค skip_log ทั้งหมด (ดูว่ามีสนาม/session ไหนดึงไม่ได้บ้าง)

In [ ]:
skip_df = pd.DataFrame(skip_log, columns=['Circuit', 'Year', 'Session', 'Reason'])
print(f"จำนวนที่ skip ทั้งหมด: {skip_df.shape[0]}")
skip_df


---
## ข้อ 3: Data Preprocessing

ขั้นตอนนี้ลบข้อมูลซ้ำและ target ที่ใช้ไม่ได้, จัดการ missing values, ทำ Target Encoding สำหรับ Driver/Circuit, One-Hot Encoding สำหรับ Team, Scaling, Feature Selection และ Train/Test Split โดยป้องกัน data leakage

- ถ้ามีข้อมูลปี 2023 จะแบ่งปี 2021-2022 เป็น train และปี 2023 เป็น test
- ถ้าไม่มีปี 2023 จะ fallback เป็น random 80/20 split
- imputer, encoder, scaler และ selector เรียนรู้จาก train set เท่านั้น

In [ ]:
from preprocessing import prepare_data

# ใช้ df_final จาก Cell 6 หรืออ่าน CSV เมื่อเปิด Notebook ใหม่
if 'df_final' not in globals():
    df_final = pd.read_csv('f1_all_circuits_raw.csv')

prepared = prepare_data(df_final, test_year=2023, max_features=15)
X_train, X_test = prepared.X_train, prepared.X_test
y_train, y_test = prepared.y_train, prepared.y_test

print(prepared.split_description)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Duplicates removed: {prepared.report["duplicate_rows_removed"]}')
print(f'Invalid targets removed: {prepared.report["invalid_target_rows_removed"]}')
print(f'Remaining missing values: {prepared.report["remaining_missing_values"]}')
print('Missing values found in raw data:')
display(pd.Series(prepared.report['missing_values_before_imputation']).loc[lambda values: values > 0])
print('Selected features:')
display(pd.Series(prepared.report['selected_features'], name='Feature'))


In [ ]:
# ตรวจสอบผลลัพธ์ก่อนนำไปสร้างโมเดล
assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)
assert list(X_train.columns) == list(X_test.columns)
assert not X_train.isna().any().any()
assert not X_test.isna().any().any()
print('Preprocessing validation passed')
display(X_train.head())


---
## ข้อ 4-5: Machine Learning Model และ Model Evaluation

Train Linear Regression และ Random Forest Regressor แล้วเปรียบเทียบด้วย MAE, MSE, RMSE และ R² บนข้อมูลปี 2023 ที่ไม่เคยใช้เรียนรู้

In [ ]:
from modeling import train_models

model_bundle = train_models(
    dataset_path='f1_all_circuits_raw.csv',
    artifact_dir='artifacts',
)
evaluation = model_bundle['metrics']
display(evaluation.style.format({'MAE': '{:.3f}', 'MSE': '{:.3f}', 'RMSE': '{:.3f}', 'R2': '{:.3f}'}))


In [ ]:
import matplotlib.pyplot as plt

ax = evaluation.plot.bar(x='Model', y='RMSE', legend=False, color=['#e10600', '#15151e'])
ax.set_title('Model comparison on the 2023 test season')
ax.set_ylabel('RMSE (seconds, lower is better)')
ax.tick_params(axis='x', rotation=0)
plt.show()


---
## ข้อ 6: Web Application

หน้า Streamlit อยู่ใน `app.py` และใช้ model artifact ที่สร้างจาก cell ด้านบน เปิดผ่าน Docker ที่ http://localhost:8501

```powershell
docker compose up -d
```

โปรเจกต์ครบตั้งแต่ Data Collection → Preprocessing → Training → Evaluation → Prediction App แล้ว